# EX 7: Text Preprocessing and Embedding Generation Using NLP Libraries

**Name:** Mohamed Aashik S

**Roll No.:** 24BAD072  

## Setup: Install and import required libraries

In [ ]:
try:
    import gensim
except ImportError:
    import sys
    import subprocess
    print("Installing 'gensim' package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gensim"])

import re
import string
import numpy as np
import nltk
from collections import Counter
from nltk.corpus import brown, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from gensim.models import Word2Vec

for resource in ["brown", "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"Could not download {resource}: {exc}")

print("NLTK, NumPy, and Gensim imports completed.")

NLTK, NumPy, and Gensim imports completed.


## Part A: Dataset Preparation and Text Preprocessing

In [ ]:
ROLL_NO = "24BAD072"
CATEGORY = "news"

raw_sentences = brown.sents(categories=CATEGORY)
print(f"Roll No.: {ROLL_NO}")
print(f"Selected Brown Corpus category: {CATEGORY}")
print(f"Number of sentences loaded: {len(raw_sentences)}")

print("\nSample original sentences:")
for i, sentence in enumerate(raw_sentences[:5], start=1):
    print(f"{i}. {' '.join(sentence)}")

Roll No.: 24BAD072
Selected Brown Corpus category: news
Number of sentences loaded: 4623

Sample original sentences:
1. The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
2. The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .
3. The September-October term jury had been charged by Fulton Superior Court Judge Durwood Pye to investigate reports of possible `` irregularities '' in the hard-fought primary which was won by Mayor-nominate Ivan Allen Jr. .
4. `` Only a relative handful of such reports was received '' , the jury said , `` considering the widespread interest in the election , the number of voters and the size of this city '' .
5. The jury said it did find that many of Georgia's registrati

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_sentence(sentence):
    text = " ".join(sentence).lower()
    tokens = word_tokenize(text)

    tokens = [token for token in tokens if token.isalpha()]

    tokens = [token for token in tokens if token not in stop_words]

    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

processed_sentences = [preprocess_sentence(sentence) for sentence in raw_sentences]
processed_sentences = [sentence for sentence in processed_sentences if sentence]

print("Original text:")
print(" ".join(raw_sentences[0]) if raw_sentences else "No sentences found.")

print("\nPreprocessed text:")
print(" ".join(processed_sentences[0]) if processed_sentences else "No processed sentences found.")

print(f"\nNon-empty preprocessed sentences: {len(processed_sentences)}")

Original text:
The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .

Preprocessed text:
fulton county grand jury said friday investigation atlanta recent primary election produced evidence irregularity took place

Non-empty preprocessed sentences: 4590


## Part B: Vocabulary Generation and Word Frequency Analysis

In [ ]:
all_tokens = [token for sentence in processed_sentences for token in sentence]

vocabulary = sorted(set(all_tokens))
frequency_distribution = FreqDist(all_tokens)

print(f"Vocabulary size (unique words): {len(vocabulary)}")
print(f"Total tokens after preprocessing: {len(all_tokens)}")

print("\nTen most frequently occurring words:")
for word, frequency in frequency_distribution.most_common(10):
    print(f"{word}: {frequency}")

print("\nSample vocabulary terms:")
print(vocabulary[:50])

print("\nFrequency distribution object created using NLTK FreqDist.")

Vocabulary size (unique words): 9896
Total tokens after preprocessing: 47682

Ten most frequently occurring words:
said: 406
year: 260
would: 249
new: 241
state: 223
one: 222
last: 177
two: 174
first: 158
president: 155

Sample vocabulary terms:
['aaron', 'abandoned', 'abandonment', 'abatuno', 'abbey', 'abe', 'abel', 'abell', 'abide', 'ability', 'ablaze', 'able', 'abner', 'aboard', 'abolish', 'abolition', 'abortive', 'abra', 'abraham', 'abroad', 'abrupt', 'absence', 'absent', 'absolute', 'absolutely', 'absorb', 'absorbed', 'abstaining', 'abstention', 'abuse', 'acacia', 'academic', 'academy', 'accardo', 'accelerated', 'accent', 'accented', 'accept', 'acceptable', 'acceptance', 'accepted', 'accepting', 'access', 'accessory', 'accident', 'accidentally', 'acclaim', 'acclaimed', 'acclimatized', 'accommodated']

Frequency distribution object created using NLTK FreqDist.


## Part C: Word Embedding Generation Using Word2Vec

In [ ]:
EMBEDDING_DIM = 100
CONTEXT_WINDOW = 5
MIN_COUNT = 2
EPOCHS = 10

if not processed_sentences or not all_tokens:
    raise ValueError("No training tokens found. Check that the Brown Corpus downloaded correctly.")

word2vec_model = Word2Vec(
    sentences=processed_sentences,
    vector_size=EMBEDDING_DIM,
    window=CONTEXT_WINDOW,
    min_count=MIN_COUNT,
    sg=1,
    workers=1,
    seed=42,
    epochs=EPOCHS
)

print("Word2Vec model training completed.")
print(f"Vocabulary learned by Word2Vec: {len(word2vec_model.wv)}")
print(f"Embedding dimension: {word2vec_model.wv.vector_size}")
print(f"Context window: {CONTEXT_WINDOW}")
print("Architecture: Skip-gram")

Word2Vec model training completed.
Vocabulary learned by Word2Vec: 5164
Embedding dimension: 100
Context window: 5
Architecture: Skip-gram


In [ ]:
if len(word2vec_model.wv) == 0:
    raise ValueError("The Word2Vec vocabulary is empty. Try lowering MIN_COUNT.")

selected_word = "government" if "government" in word2vec_model.wv else next(iter(word2vec_model.wv.key_to_index))

embedding_vector = word2vec_model.wv[selected_word]
print(f"Selected word: {selected_word}")
print(f"Embedding vector: {embedding_vector}")
print(f"Vector dimensions/shape: {embedding_vector.shape}")
print(f"Number of dimensions: {len(embedding_vector)}")

print(f"\nFive words most similar to '{selected_word}':")
similar_words = word2vec_model.wv.most_similar(selected_word, topn=min(5, len(word2vec_model.wv) - 1))
for word, score in similar_words:
    print(f"{word}: {score:.4f}")

Selected word: government
Embedding vector: [-0.03939539 -0.01844642  0.11237339  0.18734224 -0.07713591 -0.02296403
  0.23091821  0.2165161   0.1408525  -0.2891347  -0.33996132  0.01070116
 -0.01228927  0.38651592 -0.15578407 -0.22136436  0.00841898 -0.15307482
 -0.25506952  0.22462574  0.04277756  0.18880297 -0.01265004  0.06385567
 -0.00375846  0.0184399  -0.1728565   0.3096049  -0.13759232 -0.14902957
 -0.13589548 -0.11846808  0.09864164  0.11621501  0.10893569 -0.17349473
  0.19579786 -0.23722145 -0.12336223 -0.03413454  0.01806677 -0.0924122
  0.0847496  -0.21629116  0.04230527  0.01971324  0.3145833   0.26387477
  0.29094037 -0.35780343 -0.11155628 -0.22519663 -0.05564813  0.0802375
  0.04740578  0.10098162 -0.17702778 -0.00358262 -0.03867766  0.11865263
  0.00569936  0.01556148 -0.18841194  0.06478949  0.05729983  0.0842135
 -0.05820354 -0.12878728 -0.3851275  -0.08134852 -0.04647505  0.04165713
 -0.15234418 -0.16747324  0.07555635  0.04707142  0.12524785 -0.17830327
 -0.067390

## Part D: Word Similarity Analysis and Experiment Management

In [ ]:
preferred_pairs = [
    ("government", "state"),
    ("president", "minister"),
    ("city", "country"),
    ("man", "woman"),
    ("war", "peace"),
    ("school", "student"),
    ("company", "business")
]

valid_pairs = [
    pair for pair in preferred_pairs
    if pair[0] in word2vec_model.wv and pair[1] in word2vec_model.wv
]

if len(valid_pairs) < 3:
    words = list(word2vec_model.wv.key_to_index.keys())
    for i in range(len(words) - 1):
        candidate = (words[i], words[i + 1])
        if candidate not in valid_pairs and (candidate[1], candidate[0]) not in valid_pairs:
            valid_pairs.append(candidate)
        if len(valid_pairs) >= 3:
            break

selected_pairs = valid_pairs[:3]
if len(selected_pairs) < 3:
    raise ValueError("The trained vocabulary is too small to form three word pairs.")

similarity_results = []
for word1, word2 in selected_pairs:
    score = float(word2vec_model.wv.similarity(word1, word2))
    similarity_results.append((word1, word2, score))

print("Cosine similarity results:")
for word1, word2, score in similarity_results:
    print(f"({word1}, {word2}) = {score:.4f}")

most_similar_pair = max(similarity_results, key=lambda item: item[2])
print(
    f"\nMost similar selected pair: {most_similar_pair[0]} and "
    f"{most_similar_pair[1]} (cosine similarity = {most_similar_pair[2]:.4f})"
)

Cosine similarity results:
(government, state) = 0.9156
(president, minister) = 0.8229
(city, country) = 0.9096

Most similar selected pair: government and state (cosine similarity = 0.9156)


In [ ]:
print("Analysis:")
print("Cosine similarity compares the direction of two word embedding vectors.")
print("A score closer to 1 indicates more similar learned context; a score near 0 indicates weaker alignment.")
print("Interpret the scores relative to this corpus and these training settings, not as universal measures of meaning.")
print("\nObservation summary:")
for word1, word2, score in similarity_results:
    if score >= 0.5:
        interpretation = "relatively high similarity in this trained model"
    elif score >= 0.2:
        interpretation = "some contextual similarity in this trained model"
    else:
        interpretation = "weak similarity in this trained model"
    print(f"{word1} - {word2}: {score:.4f} ({interpretation})")

Analysis:
Cosine similarity compares the direction of two word embedding vectors.
A score closer to 1 indicates more similar learned context; a score near 0 indicates weaker alignment.
Interpret the scores relative to this corpus and these training settings, not as universal measures of meaning.

Observation summary:
government - state: 0.9156 (relatively high similarity in this trained model)
president - minister: 0.8229 (relatively high similarity in this trained model)
city - country: 0.9096 (relatively high similarity in this trained model)
